# Задача на табличных данных

В данной работе @Данил Растяпин и @Глеб Плаксин будут делать задачу, основанную на табличных данных 


Поставленная задача: __предсказывать риск аварии с пострадавшими на основе табличных признаков.__

Было решено взять датасет: [Cincinnati Car Crash Data](https://www.kaggle.com/datasets/steverusso/cincinnati-car-crash-data)

В качестве бизнес-заказчика можно рассматривать страховую компанию, которая занимается автострахованием и страхованием жизни. Компании важно заранее оценивать, какие ДТП с большей вероятностью приводят к травмам или смерти, так как именно такие случаи связаны с более высокими страховыми выплатами.

Мы хотим предсказывать вероятность того, что авария приведет к пострадавшим. 

Соответственно, наша задача — бинарная классификация:

- `0` — авария без пострадавших, только материальный ущерб;
- `1` — авария с пострадавшими или летальным исходом.

Для решения задачи мы будем использовать полносвязную нейронную сеть, так как данные представлены в табличном виде: каждая авария описывается набором признаков, таких как погода, освещение, тип дороги, район, тип столкновения и характеристики участников.

## Обоснование корректности применения полносвязной нейронной сети 
Небольшой дисклеймер: для решения задачи на табличных данных вполне корректно использовать полносвязную нейронную сеть, так как данные представлены в виде набора признаков, описывающих каждый объект наблюдения.

Например, x = [признак1, признак2, итд...]

Ее главное преимущество в сравнении с базовыми линейными моделями заключается в способности автоматически находить сложные нелинейные зависимости в данных, избавляя от необходимости вручную конструировать признаки. Это же __круто__!!! При всем при этом это также очень важно.



In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [61]:
df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')
df.head()

/var/folders/lk/0xgrnlvn06nf17pvwxr3n6980000gn/T/ipykernel_65534/3656284645.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')


,ADDRESS_X,LATITUDE_X,LONGITUDE_X,AGE,COMMUNITY_COUNCIL_NEIGHBORHOOD,CPD_NEIGHBORHOOD,CRASHDATE,CRASHLOCATION,CRASHSEVERITY,CRASHSEVERITYID,...,LOCALREPORTNO,MANNEROFCRASH,ROADCONDITIONSPRIMARY,ROADCONTOUR,ROADSURFACE,SNA_NEIGHBORHOOD,TYPEOFPERSON,WEATHER,ZIP,UNITTYPE
0,63XX GRACELY,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,145004877,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,45233.0,03 - MID SIZE
1,9XX CHATEAU AV,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,...,155002081,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,45204.0,02 - COMPACT
2,30XX READING RD,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,155010090,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,45206.0,04 - FULL SIZE
3,36XX READING RD,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185005525,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,45229.0,07 - PICKUP
4,37XX WARSAW AV,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185012267,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,45205.0,04 - FULL SIZE


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258672 entries, 0 to 258671
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ADDRESS_X                       258669 non-null  object 
 1   LATITUDE_X                      258672 non-null  float64
 2   LONGITUDE_X                     258672 non-null  float64
 3   AGE                             258672 non-null  object 
 4   COMMUNITY_COUNCIL_NEIGHBORHOOD  253006 non-null  object 
 5   CPD_NEIGHBORHOOD                252959 non-null  object 
 6   CRASHDATE                       258669 non-null  object 
 7   CRASHLOCATION                   194021 non-null  object 
 8   CRASHSEVERITY                   258672 non-null  object 
 9   CRASHSEVERITYID                 258672 non-null  float64
 10  DATECRASHREPORTED               258670 non-null  object 
 11  DAYOFWEEK                       258671 non-null  object 
 12  GENDER          

## Интерпретация полученных признаков
| №  | Признак                                      | Тип признака       | Что это |
|----|----------------------------------------------|--------------------|--------|
| 1  | Address_X         | Текстовый      | Адрес с маскированным домом |
| 2  | COMMUNITY_COUNCIL_NEIGHBORHOOD             | Категориальный      | Район |
| 3  | CPD_NEIGHBORHOOD         | Категориальный      | Ближайшее отделение полиции |
| 4  | CPD_NEIGHBORHOOD             | Категориальный      | Тип дороги, где произошла авария |
| 5  | CRASHSEVERITY                            | Категориальный    | Летальность аварии (сразу всмятку или лайтово) |
| 6  | INSTANCEID                                      | Числовой        | ID аварии|
| 7  | LIGHTCONDITIONSPRIMARY                      | Категориальный         | Небо во время аварии |
| 8  |  MANNEROFCRASH                           | Категориальный         |  Тип аварии |
| 9 | TYPEOFPERSON                                    | Категориальный     | Кто попал в аварию (пассажир, пешеход, водитель) |
| 10 |  UNITTYPE                             | Категориальный        | Какой вид автомобиля |

In [63]:
df.CRASHSEVERITY.unique()

array(['3 - PROPERTY DAMAGE ONLY (PDO)', '2 - INJURY', '1 - FATAL INJURY',
       '5 - PROPERTY DAMAGE ONLY', '4 - INJURY POSSIBLE',
       '3 - MINOR INJURY SUSPECTED', '2 - SERIOUS INJURY SUSPECTED',
       '1 - FATAL'], dtype=object)

In [64]:
df.INJURIES.unique()

array(['1 - NO INJURY / NONE REPORTED', '3 - NON-INCAPACITATING',
       '5 - NO APPARENTY INJURY', '4 - POSSIBLE INJURY',
       '3 - SUSPECTED MINOR INJURY', '2 - POSSIBLE', '4 - INCAPACITATING',
       '2 - SUSPECTED SERIOUS INJURY', '5 - FATAL', nan, '1 - FATAL'],
      dtype=object)

In [65]:
temp = df[['INJURIES', 'CRASHSEVERITY','ADDRESS_X']].groupby(['CRASHSEVERITY','INJURIES']).count()
temp

ADDRESS_X
CRASHSEVERITY                  INJURIES                                
1 - FATAL                      1 - FATAL                             66
                               1 - NO INJURY / NONE REPORTED          3
                               2 - SUSPECTED SERIOUS INJURY          26
                               3 - SUSPECTED MINOR INJURY            32
                               4 - POSSIBLE INJURY                   13
                               5 - FATAL                              2
                               5 - NO APPARENTY INJURY               45
1 - FATAL INJURY               1 - NO INJURY / NONE REPORTED        110
                               2 - POSSIBLE                          13
                               3 - NON-INCAPACITATING                42
                               4 - INCAPACITATING                    52
                               5 - FATAL                            172
2 - INJURY                     1 - NO INJURY / NONE REPORTED      21196
                               2 - POSSIBLE                       15297
                               3 - NON-INCAPACITATING             10317
                               4 - INCAPACITATING                  2007
                               4 - POSSIBLE INJURY                    4
                               5 - NO APPARENTY INJURY                2
2 - SERIOUS INJURY SUSPECTED   1 - NO INJURY / NONE REPORTED          8
                               2 - POSSIBLE                           5
                               2 - SUSPECTED SERIOUS INJURY         486
                               3 - NON-INCAPACITATING                 4
                               3 - SUSPECTED MINOR INJURY           154
                               4 - INCAPACITATING                     8
                               4 - POSSIBLE INJURY                   56
                               5 - NO APPARENTY INJURY              303
3 - MINOR INJURY SUSPECTED     1 - NO INJURY / NONE REPORTED          9
                               2 - SUSPECTED SERIOUS INJURY           1
                               3 - NON-INCAPACITATING                 8
                               3 - SUSPECTED MINOR INJURY          4954
                               4 - POSSIBLE INJURY                  653
                               5 - NO APPARENTY INJURY             3713
3 - PROPERTY DAMAGE ONLY (PDO) 1 - NO INJURY / NONE REPORTED     144522
                               2 - POSSIBLE                           1
                               5 - NO APPARENTY INJURY               16
4 - INJURY POSSIBLE            1 - NO INJURY / NONE REPORTED          2
                               2 - POSSIBLE                           3
                               4 - POSSIBLE INJURY                 4201
                               5 - NO APPARENTY INJURY             3584
5 - PROPERTY DAMAGE ONLY       1 - NO INJURY / NONE REPORTED         64
                               4 - POSSIBLE INJURY                    1
                               5 - NO APPARENTY INJURY            46293

In [66]:
df.columns = df.columns.str.lower()

In [67]:
temp = df[['crashseverity', 'crashseverityid']].groupby(['crashseverity','crashseverityid']).count()
temp

,
crashseverity,crashseverityid
1 - FATAL,201901.0
1 - FATAL INJURY,1.0
2 - INJURY,2.0
2 - SERIOUS INJURY SUSPECTED,201902.0
3 - MINOR INJURY SUSPECTED,201903.0
3 - PROPERTY DAMAGE ONLY (PDO),3.0
4 - INJURY POSSIBLE,201904.0
5 - PROPERTY DAMAGE ONLY,201905.0


Наверное, в какой-то момент изменилось, то, как кодируют степень тяжести нанесенного ущерба. Или для разных городов разные коды

In [68]:
# Переводим в строку без '.0' (если нет пропусков) и проверяем, что длина равна 6 цифрам
df['id6figures'] = df['crashseverityid'].dropna().astype(int).astype(str).str.len() == 6

# Заполняем пропуски (если в исходной колонке были NaN) значением False
df['id6figures'] = df['id6figures'].fillna(False).astype(int)
distribution = pd.crosstab(df['crashdate'], df['id6figures'])


In [69]:
df.drop(columns = ['zip', 'address_x',], inplace = True)

In [70]:
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,145004877,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,155002081,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,155010090,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,185005525,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,185012267,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,185005351,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,145011351,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,155001202,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,145004142,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0


### Несколько пассажиров

Также в ходе анализа было выявленно, что localreportno иногда повторяются. 

In [71]:
df[df['localreportno'] == 145001250]

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
110049,39.098891,-84.533526,51-60,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
184132,39.099991,-84.534216,51-60,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,O - OCCUPANT,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
200745,39.100121,-84.533446,18-25,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,D - DRIVER,1 - CLEAR,03 - MID SIZE,0


Например, здесь мы видим что в одной аварии участвовало три человека ———— две женщины 51-60 и мужчина 18-25, причем одна женщина и мужчина отмечены оба как "DRIVER" и они оба без травм, а вот у второй пассажирки травма "NON-INCAPACITATING", мешающая повседневной жизни. 

### Агреггируем строки аварии

Так как один localreportno может появляться несколько раз, мы можем понять, что одна строка датасета — это один из участников аварии(>=1)

Для бизнес-задачи нам нужно предсказывать тяжесть аварии целиком, поэтому дальше агрегируем данные до уровня одной аварии.

Взглянем на то, сколько строк может приходиться на одну аварию

In [72]:
df.groupby('localreportno').size().sort_values(ascending=False)

localreportno
135001864    43
185005626    33
175003760    27
175003358    26
195018224    21
             ..
145011765     1
145011763     1
145011761     1
195003901     1
165018760     1
Length: 133873, dtype: int64

In [73]:
df[df['localreportno'] == 135001864]

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
2927,39.180801,-84.519626,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
5651,39.182311,-84.520696,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
11093,39.180991,-84.519686,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
19320,39.181191,-84.519716,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
21218,39.181251,-84.519876,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
24425,39.181701,-84.520576,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
28016,39.181751,-84.520926,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
28080,39.181211,-84.520096,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
31243,39.181371,-84.521216,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
34505,39.180901,-84.521086,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0


На этом примере мы видим огромный автобус с детишками на 43 человек. Вело автобус два водителя. Взглянем на распределение 

In [74]:
import plotly.express as px

In [75]:
reports_count = df.groupby('localreportno').size().reset_index(name='rows_per_accident')
px.histogram(reports_count, x='rows_per_accident', title='строк на одну аварию')

Смертельных аварий очень мало, модель может плоховато обучиться на редком классе. Поэтому берем за таргет наличие пострадавших. 

In [76]:
df['target'] = (~df['crashseverity'].str.contains('PROPERTY DAMAGE ONLY', case=False, na=False)).astype(int)
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures,target
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0,1
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0


In [77]:
df['target'].value_counts()

target
0    191120
1     67552
Name: count, dtype: int64

Прекрасно! __67__,552 строчек аварий с травмами. И 191,120 строчек аварий без травм (только проперти дэмэдж).

Теперь аггрегируем участников аварий, чтобы мы могли детально знать кол-во пострадавших и прочие показатели


### Агрегируем количество участников
Посчитаем, сколько участников на каждую аварию

In [78]:
num_people = df.groupby('localreportno').size().reset_index(name='num_people')
num_people

,localreportno,num_people
0,14,2
1,15,1
2,30,2
3,31,3
4,1455001,1
...,...,...
133868,215000458,1
133869,215000459,1
133870,215000460,1
133871,215000461,2


In [79]:
num_people['num_people'].describe()

count    133873.000000
mean          1.932219
std           0.845384
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          43.000000
Name: num_people, dtype: float64

Посчитаем количество водителей, пассажиров и пешеходов на каждую аварию

In [80]:
num_drivers = (df.groupby('localreportno')['typeofperson'].apply(lambda x: (x == 'D - DRIVER').sum()).reset_index(name='num_drivers'))
num_drivers

,localreportno,num_drivers
0,14,2
1,15,1
2,30,2
3,31,3
4,1455001,1
...,...,...
133868,215000458,1
133869,215000459,1
133870,215000460,1
133871,215000461,2


In [81]:
num_occupants = (df.groupby('localreportno')['typeofperson'].apply(lambda x: (x == 'O - OCCUPANT').sum()).reset_index(name='num_occupants'))
num_occupants

,localreportno,num_occupants
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0
...,...,...
133868,215000458,0
133869,215000459,0
133870,215000460,0
133871,215000461,0


Был ли в аварии пешеход и сколько

In [82]:
num_pedestrians = (df.groupby('localreportno')['typeofperson'].apply(lambda x: (x == 'P - PEDESTRIAN').sum()).reset_index(name='num_pedestrians'))
num_pedestrians

,localreportno,num_pedestrians
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0
...,...,...
133868,215000458,0
133869,215000459,0
133870,215000460,0
133871,215000461,0


In [83]:
has_pedestrian = (df.groupby('localreportno')['typeofperson'].apply(lambda x: int((x == 'P - PEDESTRIAN').any())).reset_index(name='has_pedestrian'))
has_pedestrian

,localreportno,has_pedestrian
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0
...,...,...
133868,215000458,0
133869,215000459,0
133870,215000460,0
133871,215000461,0


In [84]:
has_occupant = (df.groupby('localreportno')['typeofperson'].apply(lambda x: int((x == 'O - OCCUPANT').any())).reset_index(name='has_occupant'))
has_occupant

,localreportno,has_occupant
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0
...,...,...
133868,215000458,0
133869,215000459,0
133870,215000460,0
133871,215000461,0


Теперь отдельно вщглянем на типы транспортных средств. Для страховой нужно понимать, участвовали ли в аварии мотоциклы, велосипеды, автобусы.

In [85]:
has_bus = (df.groupby('localreportno')['unittype'].apply(lambda x: int(x.astype(str).str.contains('BUS', case=False, na=False).any())).reset_index(name='has_bus'))
has_bus

,localreportno,has_bus
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0
...,...,...
133868,215000458,0
133869,215000459,0
133870,215000460,0
133871,215000461,0


In [86]:
has_motorcycle = (df.groupby('localreportno')['unittype'].apply(lambda x: int(x.astype(str).str.contains('MOTORCYCLE', case=False, na=False).any())).reset_index(name='has_motorcycle'))
has_motorcycle.head()

,localreportno,has_motorcycle
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0


In [87]:
has_bicycle = (df.groupby('localreportno')['unittype'].apply(lambda x: int(x.astype(str).str.contains('BICYCLE', case=False, na=False).any())).reset_index(name='has_bicycle'))
has_bicycle.head()

,localreportno,has_bicycle
0,14,0
1,15,0
2,30,0
3,31,0
4,1455001,0


### Собираем признаки участников ДТП в одну витрину

In [88]:
person_features = num_people.merge(num_drivers, on='localreportno', how='left')
person_features = person_features.merge(num_occupants, on='localreportno', how='left')
person_features = person_features.merge(num_pedestrians, on='localreportno', how='left')
person_features = person_features.merge(has_pedestrian, on='localreportno', how='left')
person_features = person_features.merge(has_occupant, on='localreportno', how='left')
person_features = person_features.merge(has_bus, on='localreportno', how='left')
person_features = person_features.merge(has_motorcycle, on='localreportno', how='left')
person_features = person_features.merge(has_bicycle, on='localreportno', how='left')

In [89]:
person_features

,localreportno,num_people,num_drivers,num_occupants,num_pedestrians,has_pedestrian,has_occupant,has_bus,has_motorcycle,has_bicycle
0,14,2,2,0,0,0,0,0,0,0
1,15,1,1,0,0,0,0,0,0,0
2,30,2,2,0,0,0,0,0,0,0
3,31,3,3,0,0,0,0,0,0,0
4,1455001,1,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
133868,215000458,1,1,0,0,0,0,0,0,0
133869,215000459,1,1,0,0,0,0,0,0,0
133870,215000460,1,1,0,0,0,0,0,0,0
133871,215000461,2,2,0,0,0,0,0,0,0


__лол. в 675 авариях не было водителя. интересно__

Некоторые признаки одинаковые для всех строк внутри одного `localreportno`: погода, освещение, тип столкновения, дата, район и т.д. Их можно взять по одному разу.

In [90]:
crash_cols = [
    'localreportno',
    'latitude_x',
    'longitude_x',
    'community_council_neighborhood',
    'cpd_neighborhood',
    'crashdate',
    'crashlocation',
    'crashseverity',
    'crashseverityid',
    'dayofweek',
    'lightconditionsprimary',
    'mannerofcrash',
    'roadconditionsprimary',
    'roadcontour',
    'roadsurface',
    'sna_neighborhood',
    'weather',
    'target'
]

In [91]:
df_crash = df[crash_cols].drop_duplicates('localreportno')
df_crash.head()

,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,lightconditionsprimary,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,weather,target
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,1 - DAYLIGHT,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,1 - CLEAR,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,1 - DAYLIGHT,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,1 - CLEAR,1
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,4 - DARK - LIGHTED ROADWAY,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,1 - CLEAR,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,1 - DAYLIGHT,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,1 - CLEAR,0
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,3 - DUSK,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,1 - CLEAR,0


Теперь соединим признаки самой аварии с агрегированными признаками участников.

In [92]:
df_model = df_crash.merge(person_features, on='localreportno', how='left')
df_model

,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,...,target,num_people,num_drivers,num_occupants,num_pedestrians,has_pedestrian,has_occupant,has_bus,has_motorcycle,has_bicycle
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,2,2,0,0,0,0,0,0,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,...,1,2,1,1,0,0,1,0,0,0
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,...,0,5,2,3,0,0,1,0,0,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,2,2,0,0,0,0,0,0,0
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,2,2,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133868,155016727,39.095846,-84.560807,SEDAMSVILLE,SEDAMSVILLE,11/17/2015 09:55:00 AM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,1,1,0,0,0,0,0,0,0
133869,155016340,39.099411,-84.502986,DOWNTOWN,C. B. D. / RIVERFRONT,11/09/2015 11:39:00 AM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,MON,...,0,1,1,0,0,0,0,0,0,0
133870,145004288,39.136355,-84.531823,CUF,CLIFTON/UNIVERSITY HEIGHTS,06/04/2014 07:03:00 AM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,WED,...,0,1,1,0,0,0,0,0,0,0
133871,195003855,39.138113,-84.531107,CUF,CLIFTON/UNIVERSITY HEIGHTS,03/24/2019 11:58:00 PM,NaN,5 - PROPERTY DAMAGE ONLY,201905.0,SUN,...,0,1,1,0,0,0,0,0,0,0


Теперь `df_model` — это уже финальная таблица для дальнейшего анализа и модели: одна строка соответствует одной аварии.

---

In [93]:
px.histogram(df_model, x='target', title='распределение аварий без постральных и с пострадавшими')


### Пощупаем гипотезы и поищем эвристики

In [94]:
df_model.groupby('has_pedestrian')['target'].mean()

has_pedestrian
0    0.192340
1    0.947523
Name: target, dtype: float64

<mark>То есть если в дтп участвовал пешеход, то почти в 95% случаев это дтп не относится к категории "property damage only". Лучше сидеть дома и никуда не выходить(</mark>

In [95]:
df_model.groupby('mannerofcrash')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
mannerofcrash,,
3 - HEAD-ON,2316,0.487910
1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TRANSPORT,30023,0.265896
6 - ANGLE,34585,0.259824
2 - REAR-END,33764,0.212830
"8 - SIDESWIPE, OPPOSITE DIRECTION",2872,0.168524
4 - REAR-TO-REAR,498,0.146586
"7 - SIDESWIPE, SAME DIRECTION",21161,0.077123
5 - BACKING,4880,0.043852
9 - UNKNOWN,3773,0.031540


In [96]:
px.bar(df_model.groupby('mannerofcrash')['target'].agg(['count', 'mean']).reset_index().sort_values('mean', ascending=False)
       , x='mannerofcrash', y='mean', title='доля аварий с пострадавшими в зависимости от типа столкновения')

Наибольшая доля дтп с пострадавшими по типу столкновения наблюдается при лобовом столкновении ——— ≈49%. 

Наиболее тяжёлые последствия связаны с лобовыми и угловыми ударами. А вот касательные и парковочные инциденты значительно реже приводят к пострадавшим.

Гипотеза: неблагоприятные погодные условия повышают вероятность ДТП с пострадавшими.

In [97]:
df_model.groupby('weather')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
weather,,
7 - SEVERE CROSSWINDS,27,0.296296
4 - RAIN,19440,0.223765
2 - CLOUDY,23212,0.218206
"3 - FOG, SMOG, SMOKE",190,0.205263
1 - CLEAR,86510,0.204878
"5 - SLEET, HAIL",224,0.200893
"8 - BLOWING SAND, SOIL, DIRT, SNOW",25,0.200000
6 - SNOW,2797,0.186271
"5 - SLEET,HAIL",44,0.181818


In [98]:
px.bar(df_model.groupby('weather')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False).reset_index(), 
       x='weather', y='mean', title='доля аварий с пострадавшими в зависимости от погодных условий')

Неожиданный инсайт: <mark> погода влияет слабее, чем пешеходы или тип столкновения </mark>

Разница между CLEAR, CLOUDY, RAIN, FOG небольшая. Погодные условия не выглядят очень сильным фактором тяжести ДТП.

Гипотеза: недостаточная освещённость повышает вероятность дтп с пострадавшими.

In [99]:
df_model.groupby('lightconditionsprimary')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
lightconditionsprimary,,
3 - DARK - LIGHTED ROADWAY,8600,0.255233
2 - DUSK,880,0.237500
4 - DARK - LIGHTED ROADWAY,24795,0.216939
2 - DAWN,2776,0.216859
1 - DAYLIGHT,90015,0.205921
3 - DUSK,2485,0.193964
9 - OTHER,27,0.185185
4 - DARK – ROADWAY NOT LIGHTED,593,0.182125
5 - DARK – ROADWAY NOT LIGHTIED,728,0.146978


In [100]:
px.bar(df_model.groupby('lightconditionsprimary')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False).reset_index(), 
       x='lightconditionsprimary', y='mean', title='доля аварий с пострадавшими в зависимости от освещения')

Одна и та же категория встречается с разными числовыми кодами. Пупупу. Видимо это степени


ДТП в тёмное время суток на освещённой дороге имеют более высокую долю пострадавших, чем ДТП днём.

In [101]:
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures,target
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0,1
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0


### Продолжаем приводить признаки к нормальному формату 

В данных я вижу немного странности в longtitude и latitude.


In [102]:
df[['latitude_x', 'longitude_x']].describe()

,latitude_x,longitude_x
count,2.586720e+05,2.586720e+05
mean,3.333947e+03,-1.520631e+04
std,1.656398e+06,7.684952e+06
min,-8.448932e+01,-3.908552e+09
25%,3.911692e+01,-8.454885e+01
50%,3.913638e+01,-8.451477e+01
75%,3.916082e+01,-8.448477e+01
max,8.424329e+08,1.000000e-03


In [103]:
bad_coords = (~df['latitude_x'].between(38, 41) | ~df['longitude_x'].between(-86, -83))
bad_coords.sum()

np.int64(202)

аж 202 значения. не круто 

так как таких строк мало, заменим некорректные координаты на пропуски.

In [104]:
df.loc[bad_coords, ['latitude_x', 'longitude_x']] = np.nan
df[['latitude_x', 'longitude_x']].describe()

,latitude_x,longitude_x
count,258470.000000,258470.000000
mean,39.140806,-84.515360
std,0.033307,0.053150
min,39.023221,-85.514230
25%,39.116956,-84.548848
50%,39.136391,-84.514783
75%,39.160825,-84.484805
max,40.000081,-84.103780


In [105]:
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures,target
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0,1
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0


In [106]:
df['localreportno'].nunique()

133873

In [107]:
reports_count = df.groupby('localreportno').size().reset_index(name='rows_per_accident')
reports_count['rows_per_accident']

0         2
1         1
2         2
3         3
4         1
         ..
133868    1
133869    1
133870    1
133871    2
133872    2
Name: rows_per_accident, Length: 133873, dtype: int64

In [108]:
reports_count['rows_per_accident'].value_counts().sort_index()

rows_per_accident
1     35398
2     81076
3     12109
4      3276
5      1183
6       483
7       198
8        66
9        38
10       18
11       10
12        6
13        1
14        1
15        3
19        2
21        1
26        1
27        1
33        1
43        1
Name: count, dtype: int64

проверим утечки данных

In [109]:
pd.crosstab(df['crashseverity'], df['target'])

target,0,1
crashseverity,,
1 - FATAL,0,187
1 - FATAL INJURY,0,389
2 - INJURY,0,48823
2 - SERIOUS INJURY SUSPECTED,0,1024
3 - MINOR INJURY SUSPECTED,0,9338
3 - PROPERTY DAMAGE ONLY (PDO),144709,0
4 - INJURY POSSIBLE,0,7791
5 - PROPERTY DAMAGE ONLY,46411,0


In [110]:
pd.crosstab(df['crashseverityid'], df['target'])

target,0,1
crashseverityid,,
1.0,0,389
2.0,0,48823
3.0,144709,0
201901.0,0,187
201902.0,0,1024
201903.0,0,9338
201904.0,0,7791
201905.0,46411,0


In [111]:
pd.crosstab(df['injuries'], df['target'])

target,0,1
injuries,,
1 - FATAL,0,66
1 - NO INJURY / NONE REPORTED,144588,21328
2 - POSSIBLE,1,15318
2 - SUSPECTED SERIOUS INJURY,0,513
3 - NON-INCAPACITATING,0,10371
3 - SUSPECTED MINOR INJURY,0,5140
4 - INCAPACITATING,0,2067
4 - POSSIBLE INJURY,1,4927
5 - FATAL,0,174


сrashseverity и crashseverityid задают целевую переменную, а injuries напрямую описывают наличие пострадавших. 

__Соответственно эти признаки нельзя использовать при обучении модели.__

соберем аварии 

In [112]:
df.shape

(258672, 26)

In [113]:
df_model.shape

(133873, 27)

In [114]:
df['localreportno'].nunique()

133873

In [115]:
df_model['localreportno'].nunique()

133873

Мы уже посчитали агрегированные признаки участников через transform. Эти значения одинаковы для всех строк одной аварии, 
 
Теперь можно оставить одну строку на каждый localreportno. 

Таким образом мы перейдем к детализации по авариям, а не по людям

In [116]:
age_features = df.groupby('localreportno').agg(
    has_under_18=('age', lambda x: int((x == 'UNDER 18').any())),
    has_18_25=('age', lambda x: int((x == '18-25').any())),
    has_26_30=('age', lambda x: int((x == '26-30').any())),
    has_31_40=('age', lambda x: int((x == '31-40').any())),
    has_41_50=('age', lambda x: int((x == '41-50').any())),
    has_51_60=('age', lambda x: int((x == '51-60').any())),
    has_61_70=('age', lambda x: int((x == '61-70').any())),
    has_over_70=('age', lambda x: int((x == 'OVER 70').any())),
    has_unknown_age=('age', lambda x: int((x == 'UNKNOWN').any())),
).reset_index()

In [117]:
df_model = df_model.merge(age_features, on='localreportno', how='left')
df_model.head()

,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,...,has_bicycle,has_under_18,has_18_25,has_26_30,has_31_40,has_41_50,has_51_60,has_61_70,has_over_70,has_unknown_age
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,0,0,0,1,1,0,0,0,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,...,0,0,1,0,0,0,0,0,0,0
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,...,0,0,1,1,0,0,0,0,0,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,0,0,0,0,1,0,1,0,0
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,0,0,1,1,0,0,0,0,0


In [118]:
df_model['crashdate'] = pd.to_datetime(df_model['crashdate'])

/var/folders/lk/0xgrnlvn06nf17pvwxr3n6980000gn/T/ipykernel_65534/2617359285.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_model['crashdate'] = pd.to_datetime(df_model['crashdate'])


In [119]:
df_model['hour'] = df_model['crashdate'].dt.hour
df_model['month'] = df_model['crashdate'].dt.month
df_model['year'] = df_model['crashdate'].dt.year
df_model['is_weekend'] = df_model['dayofweek'].isin(['SAT', 'SUN']).astype(int)

In [120]:
df_model

,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,...,has_31_40,has_41_50,has_51_60,has_61_70,has_over_70,has_unknown_age,hour,month,year,is_weekend
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,2014-06-17 17:25:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,1,1,0,0,0,0,17.0,6.0,2014.0,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,2015-02-15 15:00:00,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,...,0,0,0,0,0,0,15.0,2.0,2015.0,1
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,2015-07-23 23:54:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,...,0,0,0,0,0,0,23.0,7.0,2015.0,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,2018-04-21 13:00:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,1,0,1,0,0,13.0,4.0,2018.0,1
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,2018-09-01 19:59:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,1,0,0,0,0,0,19.0,9.0,2018.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133868,155016727,39.095846,-84.560807,SEDAMSVILLE,SEDAMSVILLE,2015-11-17 09:55:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,0,0,0,0,0,9.0,11.0,2015.0,0
133869,155016340,39.099411,-84.502986,DOWNTOWN,C. B. D. / RIVERFRONT,2015-11-09 11:39:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,MON,...,1,0,0,0,0,0,11.0,11.0,2015.0,0
133870,145004288,39.136355,-84.531823,CUF,CLIFTON/UNIVERSITY HEIGHTS,2014-06-04 07:03:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,WED,...,0,0,0,0,0,0,7.0,6.0,2014.0,0
133871,195003855,39.138113,-84.531107,CUF,CLIFTON/UNIVERSITY HEIGHTS,2019-03-24 23:58:00,NaN,5 - PROPERTY DAMAGE ONLY,201905.0,SUN,...,0,0,0,0,0,1,23.0,3.0,2019.0,1


__ОЧЕНЬ ХОРОШО__, мы убрали почти весь мусор

In [121]:
df_model['target'].value_counts()

target
0    106066
1     27807
Name: count, dtype: int64

In [122]:
df_model['target'].value_counts(normalize=True)

target
0    0.792288
1    0.207712
Name: proportion, dtype: float64

После агрегации одна строка соответствует одной аварии. Доля аварий с пострадавшими составляет около 21%, поэтому классы несбалансированы, но не прям критично

In [123]:
missing = df_model.isna().mean().sort_values(ascending=False)
missing

crashlocation                     0.249572
sna_neighborhood                  0.021782
cpd_neighborhood                  0.021692
community_council_neighborhood    0.021528
year                              0.000022
crashdate                         0.000022
month                             0.000022
hour                              0.000022
roadcontour                       0.000015
roadconditionsprimary             0.000015
weather                           0.000015
roadsurface                       0.000015
lightconditionsprimary            0.000015
mannerofcrash                     0.000007
dayofweek                         0.000007
has_18_25                         0.000000
has_51_60                         0.000000
has_26_30                         0.000000
has_31_40                         0.000000
has_41_50                         0.000000
localreportno                     0.000000
has_61_70                         0.000000
has_over_70                       0.000000
has_unknown

In [124]:
px.bar(missing[missing > 0].reset_index(), x='index', y=0, title='Доля пропусков по признакам')

В crashlocation больше всего пропусков. 
Остальные признаки вроде бы имеют небольшую долю пропусков...

In [125]:
df_model_clean = df_model.copy()

In [126]:
cat_cols = df_model_clean.select_dtypes(include='object').columns.tolist()
num_cols = df_model_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols

['community_council_neighborhood',
 'cpd_neighborhood',
 'crashlocation',
 'crashseverity',
 'dayofweek',
 'lightconditionsprimary',
 'mannerofcrash',
 'roadconditionsprimary',
 'roadcontour',
 'roadsurface',
 'sna_neighborhood',
 'weather']

In [127]:
num_cols

['localreportno',
 'latitude_x',
 'longitude_x',
 'crashseverityid',
 'target',
 'num_people',
 'num_drivers',
 'num_occupants',
 'num_pedestrians',
 'has_pedestrian',
 'has_occupant',
 'has_bus',
 'has_motorcycle',
 'has_bicycle',
 'has_under_18',
 'has_18_25',
 'has_26_30',
 'has_31_40',
 'has_41_50',
 'has_51_60',
 'has_61_70',
 'has_over_70',
 'has_unknown_age',
 'hour',
 'month',
 'year',
 'is_weekend']

In [128]:
num_cols = [col for col in num_cols if col != 'target']

Заполняем кат. пропуски волшебной фразой 'Unknown'

In [129]:
for col in cat_cols:
    df_model_clean[col] = df_model_clean[col].fillna('Unknown')

In [130]:
df_model_clean


,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,...,has_31_40,has_41_50,has_51_60,has_61_70,has_over_70,has_unknown_age,hour,month,year,is_weekend
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,2014-06-17 17:25:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,1,1,0,0,0,0,17.0,6.0,2014.0,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,2015-02-15 15:00:00,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,...,0,0,0,0,0,0,15.0,2.0,2015.0,1
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,2015-07-23 23:54:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,...,0,0,0,0,0,0,23.0,7.0,2015.0,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,2018-04-21 13:00:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,1,0,1,0,0,13.0,4.0,2018.0,1
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,2018-09-01 19:59:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,1,0,0,0,0,0,19.0,9.0,2018.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133868,155016727,39.095846,-84.560807,SEDAMSVILLE,SEDAMSVILLE,2015-11-17 09:55:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,0,0,0,0,0,9.0,11.0,2015.0,0
133869,155016340,39.099411,-84.502986,DOWNTOWN,C. B. D. / RIVERFRONT,2015-11-09 11:39:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,MON,...,1,0,0,0,0,0,11.0,11.0,2015.0,0
133870,145004288,39.136355,-84.531823,CUF,CLIFTON/UNIVERSITY HEIGHTS,2014-06-04 07:03:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,WED,...,0,0,0,0,0,0,7.0,6.0,2014.0,0
133871,195003855,39.138113,-84.531107,CUF,CLIFTON/UNIVERSITY HEIGHTS,2019-03-24 23:58:00,Unknown,5 - PROPERTY DAMAGE ONLY,201905.0,SUN,...,0,0,0,0,0,1,23.0,3.0,2019.0,1


In [131]:
num_cols = df_model_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col not in ['target', 'localreportno']]

In [132]:
df_model_clean[num_cols] = df_model_clean[num_cols].fillna(df_model_clean[num_cols].median())

In [133]:
df_model_clean

,localreportno,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,dayofweek,...,has_31_40,has_41_50,has_51_60,has_61_70,has_over_70,has_unknown_age,hour,month,year,is_weekend
0,145004877,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,2014-06-17 17:25:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,1,1,0,0,0,0,17.0,6.0,2014.0,0
1,155002081,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,2015-02-15 15:00:00,01 - NOT AN INTERSECTION,2 - INJURY,2.0,SUN,...,0,0,0,0,0,0,15.0,2.0,2015.0,1
2,155010090,39.135486,-84.496520,AVONDALE,AVONDALE,2015-07-23 23:54:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,THU,...,0,0,0,0,0,0,23.0,7.0,2015.0,0
3,185005525,39.147889,-84.489222,AVONDALE,AVONDALE,2018-04-21 13:00:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,0,1,0,1,0,0,13.0,4.0,2018.0,1
4,185012267,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,2018-09-01 19:59:00,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,SAT,...,1,0,0,0,0,0,19.0,9.0,2018.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133868,155016727,39.095846,-84.560807,SEDAMSVILLE,SEDAMSVILLE,2015-11-17 09:55:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,TUE,...,0,0,0,0,0,0,9.0,11.0,2015.0,0
133869,155016340,39.099411,-84.502986,DOWNTOWN,C. B. D. / RIVERFRONT,2015-11-09 11:39:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,MON,...,1,0,0,0,0,0,11.0,11.0,2015.0,0
133870,145004288,39.136355,-84.531823,CUF,CLIFTON/UNIVERSITY HEIGHTS,2014-06-04 07:03:00,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,WED,...,0,0,0,0,0,0,7.0,6.0,2014.0,0
133871,195003855,39.138113,-84.531107,CUF,CLIFTON/UNIVERSITY HEIGHTS,2019-03-24 23:58:00,Unknown,5 - PROPERTY DAMAGE ONLY,201905.0,SUN,...,0,0,0,0,0,1,23.0,3.0,2019.0,1


гуд остался только crashdate, но я его лучше дропну...

### Финальная трансформация признаков 

In [136]:
drop_cols = ['localreportno', 'crashdate', 'crashseverity',
             'crashseverityid', 'target']


In [137]:
X = df_model_clean.drop(columns=drop_cols)
y = df_model_clean['target']

In [139]:
X

,latitude_x,longitude_x,community_council_neighborhood,cpd_neighborhood,crashlocation,dayofweek,lightconditionsprimary,mannerofcrash,roadconditionsprimary,roadcontour,...,has_31_40,has_41_50,has_51_60,has_61_70,has_over_70,has_unknown_age,hour,month,year,is_weekend
0,39.107808,-84.688195,SAYLER PARK,SAYLER PARK,03 - T-INTERSECTION,TUE,1 - DAYLIGHT,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,...,1,1,0,0,0,0,17.0,6.0,2014.0,0
1,39.108110,-84.560280,EAST PRICE HILL,EAST PRICE HILL,01 - NOT AN INTERSECTION,SUN,1 - DAYLIGHT,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,...,0,0,0,0,0,0,15.0,2.0,2015.0,1
2,39.135486,-84.496520,AVONDALE,AVONDALE,01 - NOT AN INTERSECTION,THU,4 - DARK - LIGHTED ROADWAY,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,...,0,0,0,0,0,0,23.0,7.0,2015.0,0
3,39.147889,-84.489222,AVONDALE,AVONDALE,01 - NOT AN INTERSECTION,SAT,1 - DAYLIGHT,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,...,0,1,0,1,0,0,13.0,4.0,2018.0,1
4,39.110989,-84.573138,EAST PRICE HILL,EAST PRICE HILL,03 - T-INTERSECTION,SAT,3 - DUSK,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,...,1,0,0,0,0,0,19.0,9.0,2018.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133868,39.095846,-84.560807,SEDAMSVILLE,SEDAMSVILLE,01 - NOT AN INTERSECTION,TUE,1 - DAYLIGHT,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,02 - WET,2 - STRAIGHT GRADE,...,0,0,0,0,0,0,9.0,11.0,2015.0,0
133869,39.099411,-84.502986,DOWNTOWN,C. B. D. / RIVERFRONT,01 - NOT AN INTERSECTION,MON,1 - DAYLIGHT,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,4 - CURVE GRADE,...,1,0,0,0,0,0,11.0,11.0,2015.0,0
133870,39.136355,-84.531823,CUF,CLIFTON/UNIVERSITY HEIGHTS,01 - NOT AN INTERSECTION,WED,1 - DAYLIGHT,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,3 - CURVE LEVEL,...,0,0,0,0,0,0,7.0,6.0,2014.0,0
133871,39.138113,-84.531107,CUF,CLIFTON/UNIVERSITY HEIGHTS,Unknown,SUN,3 - DARK - LIGHTED ROADWAY,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,4 - CURVE GRADE,...,0,0,0,0,0,1,23.0,3.0,2019.0,1


теперь трансформируем признаки:

In [144]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [145]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [146]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
         cat_cols)])

In [147]:
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

In [150]:
X_train_ready.shape, X_test_ready.shape

((107098, 288), (26775, 288))

In [154]:
y_train.value_counts()

target
0    84852
1    22246
Name: count, dtype: int64

In [155]:
y_test.value_counts()

target
0    21214
1     5561
Name: count, dtype: int64